In [2]:
import torch
from transformers import pipeline

device_id = 0 if torch.cuda.is_available() else -1

#our model
print("Downloading our model (marinaza/ru_toxic_bert)")
my_classifier = pipeline(
    "text-classification",
    model="marinaza/ru_toxic_bert",
    tokenizer="marinaza/ru_toxic_bert",
    device=device_id
)
print("Our model successfully downloaded")

#snlp model
print("Downloading baseline model (s-nlp/russian_toxicity_classifier)")
snlp_classifier = pipeline(
    "text-classification",
    model="s-nlp/russian_toxicity_classifier",
    tokenizer="s-nlp/russian_toxicity_classifier",
    device=device_id
)
print("SNLP Model successfully downloaded")



#multilingual bert
print("Downloading multilingual BERT toxicity classifier (FredZhang7/one-for-all-toxicity-v3)")
multilingual_classifier = pipeline(
    "text-classification",
    model="FredZhang7/one-for-all-toxicity-v3",
    tokenizer="FredZhang7/one-for-all-toxicity-v3",
    device=device_id
)
print("Multilingual model successfully downloaded")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12022.86it/s]


Our model successfully downloaded


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 17272.53it/s]


SNLP Model successfully downloaded


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14028.47it/s]


Multilingual model successfully downloaded


In [3]:
import pandas as pd

#data preparation
dataset_path = './nlp/balanced_toxic_dataset.csv'

df_balanced = pd.read_csv(dataset_path)


df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)


total_len = len(df_balanced)
val_end = int(0.8 * total_len)
df_test = df_balanced.iloc[val_end:].copy()


if 'comment' in df_test.columns and 'toxic' in df_test.columns:
    df_test = df_test.rename(columns={'comment': 'text', 'toxic': 'label'})

print(f"Total rows in file: {total_len}")
print(f"Test set size (df_test): {len(df_test)} rows.")

display(df_test.head(10))

Total rows in file: 98862
Test set size (df_test): 19773 rows.


,text,label
79089,"ну пять рублей, ну 98 год. и?",0
79090,народу затуманили мозги...кругом были смотрящи...,1
79091,я что то не пойму за что поддерживать? мне воо...,1
79092,продам браслет. подьебка.первому клиенту минет...,1
79093,мой как говорил дед еби кривых сорбатых косых ...,1
79094,И леваки этого Александра грохнули. Ебанутые.\n,1
79095,если деньги завтра отправлю,0
79096,"Ты опять выходишь на связь, шизик, тебя уже ис...",1
79097,ты долбоеб если так рассуждаешь! мне больше те...,1
79098,расстрелять на хуй....,1


In [4]:
import json
import pandas as pd
import os
import re

#creating a bad word dictionary for keyword-based classifier
master_bad_words = set()

#extract bad word from json
print("Reading JSON")
try:
    with open('./nlp/bad_words_1.json', 'r', encoding='utf-8') as f:
        data = json.load(f)
        if 'stemmed_words' in data:
            master_bad_words.update(data['stemmed_words'])
except Exception as e:
    print(f"Could not read JSON: {e}")

#extract bad words from csv
print("Reading CSV")
try:
    df = pd.read_csv('./nlp/bad_words_2.csv', header=None)
    csv_words = df[0].dropna().astype(str).str.lower().str.strip().tolist()
    master_bad_words.update(csv_words)
except Exception as e:
    print(f"Could not read CSV: {e}")


#extract bad words from ts files
print("Reading TypeScript (.ts) files")
root_dir = './nlp/bad_words_3'
pattern = re.compile(r"'\s*([^']+?)\s*'")

if os.path.exists(root_dir):
    for subdir, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.ts'):
                with open(os.path.join(subdir, file), 'r', encoding='utf-8') as f:
                    content = f.read()
                    forms = pattern.findall(content)
                    master_bad_words.update(forms)
else:
    print(f"Folder {root_dir} not found. Skipping TS files")


final_cleaned_words = {word.lower().strip() for word in master_bad_words if word.strip()}

output_filename = 'final_toxic_dictionary.json'
with open(output_filename, 'w', encoding='utf-8') as f:
    json.dump(list(final_cleaned_words), f, ensure_ascii=False, indent=4)

print(f"Successfully merged {len(final_cleaned_words)} unique toxic words/forms")
print(f"Saved to: {output_filename}")

Reading JSON
Reading CSV
Reading TypeScript (.ts) files
Successfully merged 802 unique toxic words/forms
Saved to: final_toxic_dictionary.json


In [5]:
import json
import re
import pandas as pd

class KeywordToxicClassifier:
    def __init__(self, dictionary_path):
        """
        Initializes the classifier and loads the toxic words dictionary.
        """
        try:
            with open(dictionary_path, 'r', encoding='utf-8') as f:
                self.toxic_words = set(json.load(f))
        except FileNotFoundError:
            print(f"Error: File {dictionary_path} not found")
            self.toxic_words = set()

    def predict_text(self, text):
        """
        Checks a single text for toxic words.
        Returns 1 (toxic) or 0 (normal).
        """
        if not isinstance(text, str):
            # protection against empty values (NaN)
            return 0

        text = text.lower()

        # extract only words, ignoring punctuation (commas, dots)
        # \w+ matches all continuous sequences of word characters
        words_in_text = re.findall(r'\w+', text)


        for word in words_in_text:
            if word in self.toxic_words:
                return 1 # found a match ->  return 1 (toxic)

        return 0 # no matches found -> return 0 (normal)

    def predict_dataset(self, texts_series):
        """
        Helper function to predict an entire DataFrame column (Pandas Series).
        """
        return texts_series.apply(self.predict_text)



In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


texts = df_test['text'].tolist()
true_labels = df_test['label'].tolist()



print("1: Our model classifies")
my_preds_raw = my_classifier(texts, batch_size=32, truncation=True, max_length=128)
my_preds = [1 if p['label'] == 'toxic' else 0 for p in my_preds_raw]



1: Our model classifies


In [8]:
print("2: SNLP model classifies")
snlp_preds_raw = snlp_classifier(texts, batch_size=32, truncation=True, max_length=128)
snlp_preds = [1 if p['label'].lower() == 'toxic' else 0 for p in snlp_preds_raw]


2: SNLP model classifies


In [9]:
print("3: Multilingual model classifies")
multi_preds_raw = multilingual_classifier(texts, batch_size=32, truncation=True, max_length=128)
multi_preds = [1 if p['label'].lower() in ['toxic', 'label_1'] else 0 for p in multi_preds_raw]


3: Multilingual model classifies


In [10]:
print("4: Keyword-based classifier classifies")
keyword_classifier_model = KeywordToxicClassifier('final_toxic_dictionary.json')
keyword_preds = df_test['text'].apply(keyword_classifier_model.predict_text).tolist()



4: Keyword-based classifier classifies


In [11]:
print("\n Calculating metrics \n")

def get_metrics(y_true, y_pred):
    return {
        "Accuracy": round(accuracy_score(y_true, y_pred), 4),
        "F1-Score": round(f1_score(y_true, y_pred), 4),
        "Precision": round(precision_score(y_true, y_pred), 4),
        "Recall": round(recall_score(y_true, y_pred), 4)
    }

my_metrics = get_metrics(true_labels, my_preds)
snlp_metrics = get_metrics(true_labels, snlp_preds)
multi_metrics = get_metrics(true_labels, multi_preds)
keyword_metrics = get_metrics(true_labels, keyword_preds)


df_comparison = pd.DataFrame(
    [my_metrics, snlp_metrics, multi_metrics, keyword_metrics],
    index=["Our Model (Fine-Tuned)", "SNLP (Neural Baseline)", "Multilingual Model", "Keyword-based (Lexical Baseline)"]
)


display(df_comparison)




df_analysis = pd.DataFrame({
    'text': texts,
    'true_label': true_labels,
    'my_pred': my_preds,
    'snlp_pred': snlp_preds,
    'keyword_pred': keyword_preds
})
# where keyword-based classifier said non toxic and our model said toxic
smart_wins = df_analysis[
    (df_analysis['true_label'] == 1) &
    (df_analysis['keyword_pred'] == 0) &
    (df_analysis['my_pred'] == 1)
]

print(f"\n{len(smart_wins)} samples where our model got toxicity from context and keyword-based classifier failed:")
if len(smart_wins) > 0:
    display(smart_wins[['text', 'true_label']].head(10))


 Calculating metrics 



,Accuracy,F1-Score,Precision,Recall
Our Model (Fine-Tuned),0.9844,0.9846,0.9831,0.9861
SNLP (Neural Baseline),0.9686,0.9682,0.9881,0.9491
Multilingual Model,0.8514,0.8510,0.8607,0.8415
Keyword-based (Lexical Baseline),0.6503,0.4830,0.9483,0.3240



6606 samples where our model got toxicity from context and keyword-based classifier failed:


,text,true_label
2,я что то не пойму за что поддерживать? мне воо...,1
5,И леваки этого Александра грохнули. Ебанутые.\n,1
7,"Ты опять выходишь на связь, шизик, тебя уже ис...",1
10,ебнутые!,1
14,вот бедная она бедная это же надо так не любит...,1
15,посадить козла,1
19,а кто ты чтобы тебе что то говорить ты холоп е...,1
28,я бы ее всю трахнул и везде кончал,1
30,У него явно вышел новый альбом...\n,1
34,они ебанутые)),1
